Моделирование мотивов в PWM

In [1]:
pip install numpy matplotlib scipy biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.0 MB/s eta 0:00:00


In [4]:
import numpy as np
from collections import Counter

# Исходные сайты связывания
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC",
         "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC",
         "CAGGTCAGC", "CAGGTTGGC"]

L = len(sites[0])
alphabet = ['A', 'C', 'G', 'T']
nucl_to_idx = {n: i for i, n in enumerate(alphabet)}

# PFM (Position Frequency Matrix)
def build_pfm(sites):
    pfm = np.zeros((4, L), dtype=float)
    for site in sites:
        for pos, nuc in enumerate(site):
            pfm[nucl_to_idx[nuc], pos] += 1
    return pfm

pfm = build_pfm(sites)
print("PFM (без псевдосчёта):\n", pfm)

# PFM с псевдосчётом α = 0.1
alpha = 0.1
pfm_pseudo = pfm + alpha
print("\nPFM с псевдосчётом α=0.1:\n", pfm_pseudo)

# PPM и PWM, делим каждый столбец на сумму
ppm = pfm_pseudo / pfm_pseudo.sum(axis=0, keepdims=True)

bg = np.array([0.295, 0.205, 0.205, 0.295])  # A, C, G, T

# PWM: логарифм отношения
pwm = np.log2(ppm / bg[:, None])
print("\nPPM (псевдосчёт 0.1):\n", ppm)
print("\nPWM:\n", pwm)

# Максимальный и минимальный теоретический скор
max_nucl_idx = np.argmax(pwm, axis=0)
max_score = np.sum(pwm[max_nucl_idx, np.arange(L)])
max_seq = ''.join(alphabet[i] for i in max_nucl_idx)
print(f"\nМаксимальный скор: {max_score:.4f}")
print(f"Последовательность: {max_seq}")

min_nucl_idx = np.argmin(pwm, axis=0)
min_score = np.sum(pwm[min_nucl_idx, np.arange(L)])
min_seq = ''.join(alphabet[i] for i in min_nucl_idx)
print(f"\nМинимальный скор: {min_score:.4f}")
print(f"Последовательность: {min_seq}")

PFM (без псевдосчёта):
 [[ 1.  8.  1.  0.  0.  2.  7.  2.  1.]
 [ 6.  2.  1.  0.  0.  6.  0.  0.  8.]
 [ 1.  0.  8. 10.  0.  0.  3.  8.  0.]
 [ 2.  0.  0.  0. 10.  2.  0.  0.  1.]]

PFM с псевдосчётом α=0.1:
 [[ 1.1  8.1  1.1  0.1  0.1  2.1  7.1  2.1  1.1]
 [ 6.1  2.1  1.1  0.1  0.1  6.1  0.1  0.1  8.1]
 [ 1.1  0.1  8.1 10.1  0.1  0.1  3.1  8.1  0.1]
 [ 2.1  0.1  0.1  0.1 10.1  2.1  0.1  0.1  1.1]]

PPM (псевдосчёт 0.1):
 [[0.10576923 0.77884615 0.10576923 0.00961538 0.00961538 0.20192308
  0.68269231 0.20192308 0.10576923]
 [0.58653846 0.20192308 0.10576923 0.00961538 0.00961538 0.58653846
  0.00961538 0.00961538 0.77884615]
 [0.10576923 0.00961538 0.77884615 0.97115385 0.00961538 0.00961538
  0.29807692 0.77884615 0.00961538]
 [0.20192308 0.00961538 0.00961538 0.00961538 0.97115385 0.20192308
  0.00961538 0.00961538 0.10576923]]

PWM:
 [[-1.47979496  1.40062343 -1.47979496 -4.93922658 -4.93922658 -0.54690915
   1.21052054 -0.54690915 -1.47979496]
 [ 1.5166018  -0.02181811 -0.95470391